# 01 – Baseline Model Evaluation

Loads `baseline.pth`, evaluates on the validation set, prints all
segmentation metrics, saves a qualitative prediction grid, and
runs inference benchmarking.

In [ ]:
import sys, os
# Ensure project root is on the path when running from notebooks/
sys.path.insert(0, os.path.abspath('..'))

import torch
from utils import Config, set_seed, get_device, device_info, print_dict
from utils.dataset import build_dataloaders
from models.model_loader import load_model, SegmentationInference
from evaluation.metrics import compute_all_metrics, MetricAccumulator
from evaluation.benchmark import Benchmarker
from evaluation.visualization import plot_predictions

cfg = Config()
cfg.ensure_dirs()
set_seed(cfg.seed)

device = get_device()
print('Device:', device_info(device))

In [ ]:
# Load model
model = load_model(cfg, device=device)
inf   = SegmentationInference(model, device, threshold=0.5)
print(model)

In [ ]:
# Evaluate
_, val_loader = build_dataloaders(cfg)
acc = MetricAccumulator()

sample_images, sample_gt, sample_pred = [], [], []

model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        probs, preds = inf.predict(images)
        acc.update(compute_all_metrics(preds, masks))
        if len(sample_images) < 4:
            sample_images.append(images)
            sample_gt.append(masks)
            sample_pred.append(preds)

metrics = acc.mean()
print_dict(metrics, 'Baseline Evaluation Metrics')

In [ ]:
# Qualitative plot
plot_predictions(
    torch.cat(sample_images),
    torch.cat(sample_gt),
    torch.cat(sample_pred),
    cfg, n=4, show=True
)

In [ ]:
# Benchmarking
bench = Benchmarker(cfg, device)
results = bench.run(model, label='baseline_fp32')
Benchmarker.print_results(results)